In [1]:
import os
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

#### Load Raw Data

In [2]:
# ── Data paths ─────────────────────────────────────────────────────────────────
DATA_PATHS = {
    'home_team':   '../data/Train_Data/train_home_team_statistics_df.csv',
    'away_team':   '../data/Train_Data/train_away_team_statistics_df.csv',
    'home_player': '../data/Train_Data/train_home_player_statistics_df.csv',
    'away_player': '../data/Train_Data/train_away_player_statistics_df.csv',
    'scores':      '../data/Y_train.csv'
}

# ── Team features ──────────────────────────────────────────────────────────────
# Original 5-last-match sums (recent form)
TEAM_FEATURES_FORM = [
    'TEAM_SHOTS_ON_TARGET_5_last_match_sum',
    'TEAM_SHOTS_TOTAL_5_last_match_sum',          # NEW: needed for shot quality ratio
    'TEAM_SHOTS_INSIDEBOX_5_last_match_sum',       # NEW: needed for shot quality ratio
    'TEAM_GOALS_5_last_match_sum',
    'TEAM_DANGEROUS_ATTACKS_5_last_match_sum',
    'TEAM_GAME_WON_5_last_match_sum',
    'TEAM_GAME_DRAW_5_last_match_sum',
    'TEAM_GAME_LOST_5_last_match_sum',
    'TEAM_SAVES_5_last_match_sum',
    'TEAM_INJURIES_5_last_match_sum',
    'TEAM_REDCARDS_5_last_match_sum',
    'TEAM_YELLOWCARDS_5_last_match_sum',           # NEW: for discipline score
]

# NEW: Season averages for long-term quality + momentum features
TEAM_FEATURES_SEASON = [
    'TEAM_SHOTS_ON_TARGET_season_average',
    'TEAM_GOALS_season_average',
    'TEAM_BALL_POSSESSION_season_average',
    'TEAM_SUCCESSFUL_PASSES_PERCENTAGE_season_average',
    'TEAM_DANGEROUS_ATTACKS_season_average',
    'TEAM_SAVES_season_average',
]

# Combined list used for merging
TEAM_FEATURES = TEAM_FEATURES_FORM + TEAM_FEATURES_SEASON

# ── Player features ────────────────────────────────────────────────────────────
PLAYER_FEATURES = [
    'PLAYER_GOALS_5_last_match_average',
    'PLAYER_ASSISTS_5_last_match_average',
    'PLAYER_MINUTES_PLAYED_5_last_match_average',
    'PLAYER_STARTING_LINEUP_5_last_match_average',
    'PLAYER_GOALS_CONCEDED_5_last_match_average',
    'PLAYER_GOALS_season_sum',
    'PLAYER_YELLOWCARDS_5_last_match_sum',
    'PLAYER_GOALS_season_std',
    'PLAYER_PENALTIES_SCORED_season_sum',
    'PLAYER_REDCARDS_season_average',
]

In [ ]:
def build_master(raw, team_features, player_features, include_target=True):
    """
    Builds a feature-engineered master DataFrame from raw data dicts.

    Parameters
    ----------
    raw : dict with keys 'home_team', 'away_team', 'home_player', 'away_player', 'scores'
    team_features : list of team feature column names (without HOME_/AWAY_ prefix)
    player_features : list of player feature column names (without HOME_/AWAY_ prefix)
    include_target : bool
        True  → encode HOME_WINS/DRAW/AWAY_WINS into a 'target' column (training)
        False → IDs only, no target column added (test/submission)

    Returns
    -------
    pd.DataFrame ready for feature selection and model prediction
    """
    # ── Target encoding ────────────────────────────────────────────────────────
    scores = raw['scores'].copy()

    if include_target:
        conditions = [scores['HOME_WINS'] == 1, scores['DRAW'] == 1, scores['AWAY_WINS'] == 1]
        scores['target'] = np.select(conditions, [2, 1, 0], default=np.nan)

        nan_count = scores['target'].isna().sum()
        if nan_count > 0:
            print(f'⚠️  {nan_count} rows with no valid target — check scores file')

        df = scores[['ID', 'target']].copy().dropna(subset=['target'])
    else:
        df = scores[['ID']].copy()

    # ── Aggregate player stats to one row per match ────────────────────────────
    # Guard against columns that may be absent in test CSVs
    home_p_cols = [c for c in player_features if c in raw['home_player'].columns]
    away_p_cols = [c for c in player_features if c in raw['away_player'].columns]

    home_player_agg = raw['home_player'].groupby('ID')[home_p_cols].mean().reset_index()
    away_player_agg = raw['away_player'].groupby('ID')[away_p_cols].mean().reset_index()

    # ── Prefix merge helper ────────────────────────────────────────────────────
    def _prefix_merge(base_df, source_df, prefix):
        prefixed = source_df.add_prefix(prefix).rename(columns={f'{prefix}ID': 'ID'})
        return base_df.merge(prefixed, on='ID', how='left')

    team_cols = [c for c in team_features if c in raw['home_team'].columns]
    df = _prefix_merge(df, raw['home_team'][['ID'] + team_cols], 'HOME_')
    df = _prefix_merge(df, home_player_agg,                      'HOME_')
    df = _prefix_merge(df, raw['away_team'][['ID'] + team_cols], 'AWAY_')
    df = _prefix_merge(df, away_player_agg,                      'AWAY_')

    # ── Drop columns >40% missing ──────────────────────────────────────────────
    missing_pct = df.isnull().mean() * 100
    high_missing = missing_pct[missing_pct > 40].index.tolist()
    if high_missing:
        print(f'  Dropping {len(high_missing)} columns with >40% missing: {high_missing}')
        df = df.drop(columns=high_missing)

    # ── Median imputation on HOME_/AWAY_ columns ───────────────────────────────
    feat_cols = [c for c in df.columns if c.startswith('HOME_') or c.startswith('AWAY_')]
    df[feat_cols] = SimpleImputer(strategy='median').fit_transform(df[feat_cols])

    # ── DIFF & RATIO features ──────────────────────────────────────────────────
    # pd.concat avoids the DataFrame fragmentation warnings from per-column assignment
    dr_cols = {}
    for col in team_features + player_features:
        h, a = f'HOME_{col}', f'AWAY_{col}'
        if h in df.columns and a in df.columns:
            dr_cols[f'DIFF_{col}']  = df[h] - df[a]
            dr_cols[f'RATIO_{col}'] = (df[h] + 1) / (df[a] + 1)
    if dr_cols:
        df = pd.concat([df, pd.DataFrame(dr_cols, index=df.index)], axis=1)

    # ── Domain features ────────────────────────────────────────────────────────
    domain_cols = {}
    for pfx in ['HOME_', 'AWAY_']:
        won, draw = f'{pfx}TEAM_GAME_WON_5_last_match_sum', f'{pfx}TEAM_GAME_DRAW_5_last_match_sum'
        if won in df.columns and draw in df.columns:
            domain_cols[f'{pfx}FORM_POINTS'] = df[won] * 3 + df[draw]

        goals, shots = f'{pfx}TEAM_GOALS_5_last_match_sum', f'{pfx}TEAM_SHOTS_ON_TARGET_5_last_match_sum'
        if goals in df.columns and shots in df.columns:
            domain_cols[f'{pfx}CONVERSION_RATE'] = (df[goals] + 1) / (df[shots] + 1)

        ib, total = f'{pfx}TEAM_SHOTS_INSIDEBOX_5_last_match_sum', f'{pfx}TEAM_SHOTS_TOTAL_5_last_match_sum'
        if ib in df.columns and total in df.columns:
            domain_cols[f'{pfx}SHOT_QUALITY_RATIO'] = (df[ib] + 1) / (df[total] + 1)

        yellow, red = f'{pfx}TEAM_YELLOWCARDS_5_last_match_sum', f'{pfx}TEAM_REDCARDS_5_last_match_sum'
        if yellow in df.columns and red in df.columns:
            domain_cols[f'{pfx}DISCIPLINE_SCORE'] = df[yellow] + df[red] * 3

        for stat in ['TEAM_GOALS', 'TEAM_SHOTS_ON_TARGET', 'TEAM_DANGEROUS_ATTACKS']:
            s_col, r_col = f'{pfx}{stat}_season_average', f'{pfx}{stat}_5_last_match_sum'
            if s_col in df.columns and r_col in df.columns:
                domain_cols[f'{pfx}{stat}_MOMENTUM'] = (df[r_col] / 5) - df[s_col]

    if domain_cols:
        df = pd.concat([df, pd.DataFrame(domain_cols, index=df.index)], axis=1)

    # ── DIFF for domain features ───────────────────────────────────────────────
    domain_diff_cols = {}
    for col in ['FORM_POINTS', 'CONVERSION_RATE', 'SHOT_QUALITY_RATIO', 'DISCIPLINE_SCORE',
                'TEAM_GOALS_MOMENTUM', 'TEAM_SHOTS_ON_TARGET_MOMENTUM', 'TEAM_DANGEROUS_ATTACKS_MOMENTUM']:
        h, a = f'HOME_{col}', f'AWAY_{col}'
        if h in df.columns and a in df.columns:
            domain_diff_cols[f'DIFF_{col}'] = df[h] - df[a]
    if domain_diff_cols:
        df = pd.concat([df, pd.DataFrame(domain_diff_cols, index=df.index)], axis=1)

    print(f'✓ build_master: {df.shape[0]} rows × {df.shape[1]} cols')
    return df


print('✓ build_master defined')

In [3]:
raw = {key: pd.read_csv(path) for key, path in DATA_PATHS.items()}

for key, df in raw.items():
    print(f'  {key}: {df.shape}')

  home_team: (12303, 143)
  away_team: (12303, 143)
  home_player: (237079, 307)
  away_player: (236132, 307)
  scores: (12303, 4)


#### Build Master Dataframe

In [ ]:
master_df = build_master(raw, TEAM_FEATURES, PLAYER_FEATURES)
master_df.head(3)

#### Define Feature Groups & Model Configs

In [ ]:
# ── Feature groups ─────────────────────────────────────────────────────────────
non_features   = {'ID', 'target'}
home_away_cols = [c for c in master_df.columns if ('HOME_' in c or 'AWAY_' in c) and c not in non_features]
diff_cols      = [c for c in master_df.columns if c.startswith('DIFF_')]
ratio_cols     = [c for c in master_df.columns if c.startswith('RATIO_')]

# Domain features only (the newly engineered columns)
domain_diff_cols = [c for c in diff_cols if any(
    x in c for x in ['FORM_POINTS', 'CONVERSION_RATE', 'SHOT_QUALITY_RATIO',
                      'DISCIPLINE_SCORE', 'MOMENTUM']
)]

ALL_FEATURE_SETS = {
    'Raw Only (Type 1)':               home_away_cols,
    'Raw + Ratio (Type 1+3)':          home_away_cols + ratio_cols,
    'Raw + Diff + Ratio (All)':        home_away_cols + diff_cols + ratio_cols,
    'Engineered Only (Diff + Ratio)':  diff_cols + ratio_cols,
    'Domain Diff Only':                domain_diff_cols,                          # NEW
    'Raw + Domain Diff':               home_away_cols + domain_diff_cols,         # NEW
}

# ACTIVE_FEATURE_SETS = [
#     'Raw Only (Type 1)',
#     'Raw + Ratio (Type 1+3)',
#     'Raw + Domain Diff',         # recommended starting point
# ]

feature_experiments = {k: v for k, v in ALL_FEATURE_SETS.items() if k in ACTIVE_FEATURE_SETS}
print(f'✓ Feature sets active: {list(feature_experiments.keys())}')
for name, cols in feature_experiments.items():
    print(f'  {name}: {len(cols)} features')
print(f'✓ feature_engineering_v2 loaded — master_df: {master_df.shape}, '
      f'feature sets: {list(feature_experiments.keys())}')

✓ Feature sets active: ['Raw Only (Type 1)', 'Raw + Ratio (Type 1+3)', 'Raw + Domain Diff']
  Raw Only (Type 1): 70 features
  Raw + Ratio (Type 1+3): 98 features
  Raw + Domain Diff: 77 features
